# Dataset Construction

Full pipeline used to produce the `dakoblov/Hallucinations` dataset from the original ToolACE corpus. This is the long-form companion to section 3.3 of `final_notebook.ipynb` and inlines every module from `src/` so the construction is reproducible from this single file. Running it end-to-end requires an `OPENROUTER_API_KEY` and approximately \$6 of OpenRouter spend for the three LLM-injection stages (~30–40 minutes wall-clock with 15 parallel workers). The published HF dataset is identical to the output of this notebook.

Pipeline stages:
1. **Type 1 (contradiction) — rule-based.** Substring substitution over five strategies. Covers ~73% of triples with exact spans.
2. **Type 1 — LLM fallback.** Qwen3-235B fills the remaining ~27%. Combined coverage 99.8%.
3. **Type 2 (overgeneration) — LLM-only.** Qwen3-235B appends a declarative sentence with a fabricated fact.
4. **Type 3 (missing tool) — LLM-only.** Qwen3-235B reads the available-tools list and appends an action-offer that requires a tool outside that list.
5. **Train augmentation.** Up to five distinct Type 1 injections per source triple.
6. **Combine + clean.** Per-type splits balanced with clean samples and concatenated with type-prefixed IDs.

In [ ]:
!pip install -q huggingface_hub requests tqdm

## API key

Set `OPENROUTER_API_KEY` for the LLM stages.

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["OPENROUTER_API_KEY"] = UserSecretsClient().get_secret("OPENROUTER_API_KEY")
except Exception:
    try:
        from google.colab import userdata
        os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
    except Exception:
        pass
assert os.environ.get("OPENROUTER_API_KEY"), "Set OPENROUTER_API_KEY for the LLM stages"

## 1. ToolACE loading and triple extraction

Download the original ToolACE `data.json` from Hugging Face. Walk every dialogue: a *triple* is any `(user_query, tool_output, assistant_final_response)` where the tool turn is immediately followed by a natural-language reply (not another tool call). Tool outputs are parsed as JSON (100% valid in ToolACE) and the list of available tools is extracted from the `system` prompt via `JSONDecoder.raw_decode`. From 11,300 ToolACE dialogues this yields 1,034 usable triples.

In [ ]:
import json
import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Iterable
from collections import defaultdict
from huggingface_hub import hf_hub_download

import json
import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Iterable

from huggingface_hub import hf_hub_download


TOOLACE_REPO = "Team-ACE/ToolACE"
TOOLACE_FILE = "data.json"


@dataclass
class Triple:
    id: str
    user: str
    tool_call: str
    tool_output_raw: str
    tool_output: list[dict[str, Any]]
    assistant: str
    tools_available: list[dict[str, Any]] = field(default_factory=list)

    def to_json(self) -> dict[str, Any]:
        return {
            "id": self.id,
            "user": self.user,
            "tool_call": self.tool_call,
            "tool_output_raw": self.tool_output_raw,
            "tool_output": self.tool_output,
            "assistant": self.assistant,
            "tools_available": self.tools_available,
        }


def download_toolace(cache_dir: str | None = None) -> Path:
    path = hf_hub_download(
        repo_id=TOOLACE_REPO,
        filename=TOOLACE_FILE,
        repo_type="dataset",
        cache_dir=cache_dir,
    )
    return Path(path)


def _is_tool_call(text: str) -> bool:
    stripped = text.lstrip()
    return stripped.startswith("[") and "(" in stripped and not stripped.startswith("[{")


def _parse_tools_list(system_prompt: str) -> list[dict[str, Any]]:
    marker = re.search(r"invoke:\s*", system_prompt)
    if not marker:
        return []
    start = marker.end()
    while start < len(system_prompt) and system_prompt[start] != "[":
        start += 1
    if start >= len(system_prompt):
        return []
    try:
        obj, _ = json.JSONDecoder().raw_decode(system_prompt, idx=start)
    except json.JSONDecodeError:
        return []
    return obj if isinstance(obj, list) else []


def extract_triples(toolace_json_path: Path | str) -> list[Triple]:
    with open(toolace_json_path) as f:
        data = json.load(f)

    triples: list[Triple] = []
    next_id = 0
    for item in data:
        conv = item.get("conversations", [])
        tools = _parse_tools_list(item.get("system", ""))
        for i in range(len(conv) - 1):
            if conv[i].get("from") != "tool" or conv[i + 1].get("from") != "assistant":
                continue
            asst_text = conv[i + 1].get("value", "")
            if _is_tool_call(asst_text) or len(asst_text.strip()) <= 20:
                continue

            user_q = None
            tool_call = ""
            for j in range(i - 1, -1, -1):
                role = conv[j].get("from")
                if role == "user" and user_q is None:
                    user_q = conv[j].get("value", "")
                    break
                if role == "assistant" and not tool_call and _is_tool_call(conv[j].get("value", "")):
                    tool_call = conv[j].get("value", "")
            if not user_q:
                continue

            tool_out_raw = conv[i].get("value", "")
            try:
                tool_out_parsed = json.loads(tool_out_raw)
            except json.JSONDecodeError:
                continue
            if not isinstance(tool_out_parsed, list):
                continue

            triples.append(
                Triple(
                    id=f"toolace-{next_id:05d}",
                    user=user_q,
                    tool_call=tool_call,
                    tool_output_raw=tool_out_raw,
                    tool_output=tool_out_parsed,
                    assistant=asst_text,
                    tools_available=tools,
                )
            )
            next_id += 1

    return triples


def write_jsonl(items: Iterable[dict[str, Any]], out_path: Path | str) -> None:
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w") as f:
        for item in items:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")


def read_jsonl(path: Path | str) -> list[dict[str, Any]]:
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]


def walk_leaves(obj: Any, path: tuple[str, ...] = ()) -> Iterable[tuple[tuple[str, ...], Any]]:
    """Yield (path, leaf_value) pairs from a nested JSON-like object."""
    if isinstance(obj, dict):
        for k, v in obj.items():
            yield from walk_leaves(v, path + (k,))
    elif isinstance(obj, list):
        for v in obj:
            yield from walk_leaves(v, path)
    else:
        yield path, obj

In [ ]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

toolace_path = download_toolace()
triples = extract_triples(toolace_path)
print(f"Extracted {len(triples)} triples from ToolACE")

write_jsonl((t.to_json() for t in triples), DATA_DIR / "triples.jsonl")

## 2. Value pools

For each `(tool_name, last_json_key)`, aggregate all short string leaf values seen in the corpus. The *cross-sample* pool spans the whole dataset and is the source for fallback swaps on scalar categorical fields. An *in-sample* pool is built per-triple in step 3 below. A blacklist of 11 'dirty' field names (`name, title, id, description, …`) excludes fields where values from different domains would mix (e.g. `condition` aggregates Used-Like-New and Partly Cloudy across e-commerce and weather APIs).

In [ ]:
import json
from collections import defaultdict
from typing import Any



DIRTY_FIELDS: frozenset[str] = frozenset({
    "name", "title", "id", "description", "text", "symbol",
    "code", "condition", "type", "status", "category",
})


def is_dirty_field(field_name: str) -> bool:
    return field_name.lower() in DIRTY_FIELDS


def build_cross_sample_pool(triples: list[Triple]) -> dict[tuple[str, str], set[str]]:
    """For each (tool_name, last_key), aggregate all string leaf values seen across the dataset."""
    pool: dict[tuple[str, str], set[str]] = defaultdict(set)
    for t in triples:
        for entry in t.tool_output:
            if not isinstance(entry, dict) or "name" not in entry:
                continue
            tool_name = entry["name"]
            results = entry.get("results", entry)
            for path, value in walk_leaves(results):
                if not path:
                    continue
                field = path[-1]
                if isinstance(value, str) and 0 < len(value) < 80:
                    pool[(tool_name, field)].add(value)
    return pool


def in_sample_pool(tool_output: list[dict[str, Any]]) -> dict[tuple[str, str], set[str]]:
    """Pool restricted to the current sample's tool_output (catches list-internal values)."""
    pool: dict[tuple[str, str], set[str]] = defaultdict(set)
    for entry in tool_output:
        if not isinstance(entry, dict) or "name" not in entry:
            continue
        tool_name = entry["name"]
        results = entry.get("results", entry)
        for path, value in walk_leaves(results):
            if not path:
                continue
            field = path[-1]
            if isinstance(value, str) and 0 < len(value) < 80:
                pool[(tool_name, field)].add(value)
    return pool

In [ ]:
cross_pool = build_cross_sample_pool(triples)
print(f"Cross-sample pool: {len(cross_pool)} (tool, field) keys")

## 3. Type 1: rule-based injection

Five strategies are tried in priority order for each triple:

1. **In-sample pool swap** — values from a list-internal sibling object in the same tool output.
2. **Type-aware int / float swap** — boundary-matched numeric substitution with 3–5× rescale.
3. **Type-aware URL swap** — modify path tail or `v=` parameter.
4. **Type-aware date swap** — try seven natural-language date formats, substitute in-format.
5. **Cross-sample pool swap** — fallback for scalar categorical strings in repeating tools.

Booleans are excluded from rule-based (94.8% paraphrased: `success:false → "failed"`). A small set of common bool-rendering regexes (e.g. `\(Correct\)→\(Incorrect\)`) catches the residual cases.

In [ ]:
import re
from dataclasses import dataclass
from datetime import date, timedelta


ISO_RE = re.compile(r"^(\d{4})-(\d{2})-(\d{2})(?:[T ](\d{2}):(\d{2}))?")

MONTH_LONG = ["", "January", "February", "March", "April", "May", "June",
              "July", "August", "September", "October", "November", "December"]
MONTH_SHORT = ["", "Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]


def parse_iso(value: str) -> date | None:
    m = ISO_RE.match(value)
    if not m:
        return None
    try:
        return date(int(m.group(1)), int(m.group(2)), int(m.group(3)))
    except ValueError:
        return None


def _ordinal_suffix(d: int) -> str:
    if 11 <= d % 100 <= 13:
        return "th"
    return {1: "st", 2: "nd", 3: "rd"}.get(d % 10, "th")


@dataclass(frozen=True)
class DateFormat:
    """A rendered form of a date and the inverse — how to render any other date in the same form."""
    name: str
    render: callable  # date -> str

    def __call__(self, d: date) -> str:
        return self.render(d)


def _render_iso(d: date) -> str:
    return d.isoformat()


def _render_month_d_yyyy(d: date) -> str:
    return f"{MONTH_LONG[d.month]} {d.day}, {d.year}"


def _render_month_dth_yyyy(d: date) -> str:
    return f"{MONTH_LONG[d.month]} {d.day}{_ordinal_suffix(d.day)}, {d.year}"


def _render_short_month_d_yyyy(d: date) -> str:
    return f"{MONTH_SHORT[d.month]} {d.day}, {d.year}"


def _render_month_d_no_year(d: date) -> str:
    return f"{MONTH_LONG[d.month]} {d.day}"


def _render_month_dth_no_year(d: date) -> str:
    return f"{MONTH_LONG[d.month]} {d.day}{_ordinal_suffix(d.day)}"


def _render_month_yyyy(d: date) -> str:
    return f"{MONTH_LONG[d.month]} {d.year}"


def _render_dth_month_yyyy(d: date) -> str:
    return f"{d.day}{_ordinal_suffix(d.day)} {MONTH_LONG[d.month]} {d.year}"


# Order matters: longest/most specific first so we prefer e.g. "January 15, 2024" over "January 15".
FORMATS: list[DateFormat] = [
    DateFormat("ISO", _render_iso),
    DateFormat("Month Dth, YYYY", _render_month_dth_yyyy),
    DateFormat("Month D, YYYY", _render_month_d_yyyy),
    DateFormat("ShortMonth D, YYYY", _render_short_month_d_yyyy),
    DateFormat("Dth Month YYYY", _render_dth_month_yyyy),
    DateFormat("Month Dth", _render_month_dth_no_year),
    DateFormat("Month D", _render_month_d_no_year),
    DateFormat("Month YYYY", _render_month_yyyy),
]


def find_in_text(d: date, text: str) -> tuple[DateFormat, int, str] | None:
    """Return (format, char_offset, rendered_string) of the first matching rendering of d in text.

    Searches formats in priority order; returns the first occurrence found.
    """
    for fmt in FORMATS:
        rendered = fmt(d)
        idx = text.find(rendered)
        if idx >= 0:
            return fmt, idx, rendered
    return None

In [ ]:
import random
import re
from dataclasses import dataclass
from datetime import timedelta
from typing import Any



@dataclass
class Span:
    start: int
    end: int
    text: str             # the new (hallucinated) substring inserted into the answer
    original_text: str    # what was there before
    field: str            # last JSON key
    strategy: str         # which strategy produced it


def _swap_substring(text: str, old: str, new: str, occurrence: int = 0) -> tuple[str, int] | None:
    """Replace the (occurrence-th) occurrence of `old` in `text` with `new`. Returns (new_text, start) or None."""
    start = -1
    pos = 0
    for _ in range(occurrence + 1):
        start = text.find(old, pos)
        if start < 0:
            return None
        pos = start + 1
    return text[:start] + new + text[start + len(old):], start


def _word_bounded_find(text: str, token: str) -> int:
    """Return the index of the first occurrence of `token` in `text` such that the surrounding
    characters are not digits or '.' (so '12' won't match inside '120' or '12.5')."""
    for m in re.finditer(re.escape(token), text):
        i, j = m.start(), m.end()
        left_ok = i == 0 or (not text[i - 1].isdigit() and text[i - 1] != ".")
        right_ok = j == len(text) or (not text[j].isdigit() and text[j] != ".")
        if left_ok and right_ok:
            return i
    return -1


def _try_int_swap(value: int, answer: str, field: str, rng: random.Random) -> tuple[str, Span] | None:
    candidates = [str(value), f"{value:,}"] if abs(value) >= 1000 else [str(value)]
    chosen = None
    for c in candidates:
        if _word_bounded_find(answer, c) >= 0:
            chosen = c
            break
    if chosen is None:
        return None

    for _ in range(30):
        if abs(value) < 10:
            new = value + rng.choice([-5, -3, -2, 2, 3, 5, 7])
        elif abs(value) < 100:
            new = value + rng.choice([-50, -30, 20, 30, 50, 70])
        else:
            factor = rng.uniform(2.5, 5.0) if rng.random() < 0.5 else rng.uniform(0.1, 0.4)
            new = int(value * factor)
        if new != value:
            break
    else:
        return None
    new_s = f"{new:,}" if "," in chosen else str(new)
    if new_s == chosen:
        return None
    swapped = _swap_substring(answer, chosen, new_s)
    if not swapped:
        return None
    new_text, start = swapped
    return new_text, Span(start=start, end=start + len(new_s), text=new_s,
                          original_text=chosen, field=field, strategy="type_based_int")


def _try_float_swap(value: float, answer: str, field: str, rng: random.Random) -> tuple[str, Span] | None:
    # Try representations in priority order (most specific first, to avoid e.g. "0.00" matching inside "0.002").
    raw = repr(value).rstrip("0").rstrip(".") if isinstance(value, float) else str(value)
    candidates = [str(value), raw, f"{value:.4f}", f"{value:.3f}", f"{value:.2f}", f"{value:.1f}"]
    seen = set()
    ordered: list[str] = []
    for c in candidates:
        if c not in seen:
            seen.add(c)
            ordered.append(c)

    chosen = None
    for c in ordered:
        # require c to be a "word"-bounded match: not just a substring of a longer numeric token
        for m in re.finditer(re.escape(c), answer):
            i, j = m.start(), m.end()
            if (i == 0 or not answer[i - 1].isdigit() and answer[i - 1] != ".") and \
               (j == len(answer) or not answer[j].isdigit() and answer[j] != "."):
                chosen = c
                break
        if chosen is not None:
            break
    if chosen is None:
        return None

    decimals = len(chosen.split(".")[1]) if "." in chosen else 0
    for _ in range(30):
        factor = rng.uniform(2.5, 5.0) if rng.random() < 0.5 else rng.uniform(0.1, 0.4)
        new = round(value * factor, decimals) if decimals else int(round(value * factor))
        new_s = f"{new:.{decimals}f}" if decimals else str(new)
        if new_s != chosen:
            break
    else:
        return None

    swapped = _swap_substring(answer, chosen, new_s)
    if not swapped:
        return None
    new_text, start = swapped
    return new_text, Span(start=start, end=start + len(new_s), text=new_s,
                          original_text=chosen, field=field, strategy="type_based_float")


def _try_date_swap(value: str, answer: str, field: str, rng: random.Random) -> tuple[str, Span] | None:
    d = parse_iso(value)
    if d is None:
        return None
    hit = find_date_in_text(d, answer)
    if hit is None:
        return None
    fmt, idx, rendered = hit
    # shift by 30..365 days, random sign
    shift = rng.randint(30, 365) * rng.choice([-1, 1])
    try:
        new_d = d + timedelta(days=shift)
    except OverflowError:
        return None
    new_rendered = fmt(new_d)
    if new_rendered == rendered:
        return None
    new_text = answer[:idx] + new_rendered + answer[idx + len(rendered):]
    return new_text, Span(start=idx, end=idx + len(new_rendered), text=new_rendered,
                          original_text=rendered, field=field, strategy=f"type_based_date[{fmt.name}]")


def _try_url_swap(value: str, answer: str, field: str, rng: random.Random) -> tuple[str, Span] | None:
    if value not in answer:
        return None
    # Modify final path segment or v= param
    new = value
    m = re.search(r"v=([\w-]+)", value)
    if m:
        new_id = "".join(rng.choice("abcdefghijklmnopqrstuvwxyz0123456789") for _ in range(len(m.group(1))))
        new = value[:m.start(1)] + new_id + value[m.end(1):]
    elif "/" in value:
        idx = value.rfind("/")
        tail = value[idx + 1:] or "page"
        new_tail = "".join(rng.choice("abcdefghijklmnopqrstuvwxyz0123456789") for _ in range(max(len(tail), 6)))
        new = value[:idx + 1] + new_tail
    if new == value:
        return None
    swapped = _swap_substring(answer, value, new)
    if not swapped:
        return None
    new_text, start = swapped
    return new_text, Span(start=start, end=start + len(new), text=new,
                          original_text=value, field=field, strategy="type_based_url")


def _try_pool_swap(value: str, answer: str, field: str, pool: set[str], rng: random.Random,
                   strategy: str = "in_sample_pool") -> tuple[str, Span] | None:
    if is_dirty_field(field):
        return None
    idx = answer.find(value)
    if idx < 0:
        return None
    alternatives = [x for x in pool if x != value and x]
    rng.shuffle(alternatives)
    # Prefer alternatives that don't already appear immediately adjacent (avoid "React, React"-style duplicates).
    for new in alternatives:
        new_text = answer[:idx] + new + answer[idx + len(value):]
        window = new_text[max(0, idx - len(new) - 4): idx + 2 * len(new) + 4]
        if window.count(new) <= 1:
            return new_text, Span(start=idx, end=idx + len(new), text=new,
                                  original_text=value, field=field, strategy=strategy)
    if alternatives:
        new = alternatives[0]
        new_text = answer[:idx] + new + answer[idx + len(value):]
        return new_text, Span(start=idx, end=idx + len(new), text=new,
                              original_text=value, field=field, strategy=strategy)
    return None


def _is_url(s: str) -> bool:
    return isinstance(s, str) and (s.startswith("http://") or s.startswith("https://"))


# Bool rewrite rules: list of (compiled_regex_for_finding, replacement_string).
# Replacement substitutes the entire match; the span returned points at the new text.
_BOOL_RULES_TRUE: dict[str, list[tuple[re.Pattern[str], str]]] = {
    "is_correct": [(re.compile(r"\(Correct\)"), "(Incorrect)")],
    "iscorrect":  [(re.compile(r"\(Correct\)"), "(Incorrect)")],
    "valid":      [(re.compile(r"\bis valid\b"), "is invalid"),
                   (re.compile(r"\bvalidated\b"), "invalidated")],
    "isvalid":    [(re.compile(r"\bis valid\b"), "is invalid")],
    "is_valid":   [(re.compile(r"\bis valid\b"), "is invalid")],
    "exists":     [(re.compile(r"\bdoes indeed exist\b"), "does not exist"),
                   (re.compile(r"\bdoes exist\b"), "does not exist")],
    "success":    [(re.compile(r"\bhas been successfully\b"), "has failed to be"),
                   (re.compile(r"\bsuccessfully\b"), "unsuccessfully")],
    "stock":          [(re.compile(r"\*\*In Stock:?\*\*:?\s*Yes\b"), "**In Stock:** No")],
    "free_shipping":  [(re.compile(r"\*\*Free Shipping\*\*:?\s*Yes\b"), "**Free Shipping**: No")],
    "prime_eligible": [(re.compile(r"\*\*Prime Eligible\*\*:?\s*Yes\b"), "**Prime Eligible**: No")],
}
_BOOL_RULES_FALSE: dict[str, list[tuple[re.Pattern[str], str]]] = {
    "valid":     [(re.compile(r"\bis not valid\b"), "is valid"),
                  (re.compile(r"\bis invalid\b"), "is valid"),
                  (re.compile(r"\binvalid\b"), "valid")],
    "isvalid":   [(re.compile(r"\bis not valid\b"), "is valid"),
                  (re.compile(r"\bis invalid\b"), "is valid")],
    "is_valid":  [(re.compile(r"\bis not valid\b"), "is valid"),
                  (re.compile(r"\bis invalid\b"), "is valid")],
    "stock":          [(re.compile(r"\*\*In Stock:?\*\*:?\s*No\b"), "**In Stock:** Yes")],
    "free_shipping":  [(re.compile(r"\*\*Free Shipping\*\*:?\s*No\b"), "**Free Shipping**: Yes")],
    "prime_eligible": [(re.compile(r"\*\*Prime Eligible\*\*:?\s*No\b"), "**Prime Eligible**: Yes")],
}


def _try_bool_swap(value: bool, answer: str, field: str) -> tuple[str, Span] | None:
    rules_map = _BOOL_RULES_TRUE if value else _BOOL_RULES_FALSE
    rules = rules_map.get(field.lower())
    if not rules:
        return None
    for pat, new_str in rules:
        m = pat.search(answer)
        if m is None:
            continue
        start, end = m.span()
        original = answer[start:end]
        if original == new_str:
            continue
        new_text = answer[:start] + new_str + answer[end:]
        return new_text, Span(start=start, end=start + len(new_str), text=new_str,
                              original_text=original, field=field, strategy="type_based_bool")
    return None


def collect_all_swaps(triple: Triple, rng: random.Random,
                      cross_pool: dict[tuple[str, str], set[str]] | None = None
                      ) -> list[tuple[str, Span]]:
    """Try every applicable strategy on every leaf value in tool_output.

    Returns a list of (corrupted_answer, span) — one entry per distinct valid swap candidate.
    De-duplicates by (start, end, text). Caller is responsible for picking among them.
    """
    answer = triple.assistant
    sample_pool = in_sample_pool(triple.tool_output)
    cross_pool = cross_pool or {}
    candidates: list[tuple[str, Span]] = []
    seen: set[tuple[int, int, str]] = set()

    for entry in triple.tool_output:
        if not isinstance(entry, dict) or "name" not in entry:
            continue
        tool_name = entry["name"]
        results_node = entry.get("results", entry)
        for path, value in walk_leaves(results_node):
            if not path:
                continue
            field = path[-1]
            attempts: list[tuple[str, Span]] = []

            if isinstance(value, bool):
                r = _try_bool_swap(value, answer, field)
                if r is not None: attempts.append(r)
            elif isinstance(value, int):
                r = _try_int_swap(value, answer, field, rng)
                if r is not None: attempts.append(r)
            elif isinstance(value, float):
                r = _try_float_swap(value, answer, field, rng)
                if r is not None: attempts.append(r)
            elif isinstance(value, str):
                if _is_url(value):
                    r = _try_url_swap(value, answer, field, rng)
                    if r is not None: attempts.append(r)
                elif parse_iso(value) is not None:
                    r = _try_date_swap(value, answer, field, rng)
                    if r is not None: attempts.append(r)
                else:
                    r = _try_pool_swap(value, answer, field,
                                       sample_pool.get((tool_name, field), set()),
                                       rng, strategy="in_sample_pool")
                    if r is not None: attempts.append(r)
                    r = _try_pool_swap(value, answer, field,
                                       cross_pool.get((tool_name, field), set()),
                                       rng, strategy="cross_sample_pool")
                    if r is not None: attempts.append(r)

            for new_text, span in attempts:
                key = (span.start, span.end, span.text)
                if key in seen:
                    continue
                seen.add(key)
                candidates.append((new_text, span))
    return candidates


def inject(triple: Triple, rng: random.Random,
           cross_pool: dict[tuple[str, str], set[str]] | None = None) -> tuple[str, list[Span]] | None:
    """Try strategies in priority order, return (corrupted_answer, [span]) or None if nothing applies.

    `cross_pool` is a global `(tool_name, field) → {values}` pool built across the whole dataset;
    used as a last-resort fallback for categorical strings when in-sample pool is exhausted.
    """
    answer = triple.assistant
    sample_pool = in_sample_pool(triple.tool_output)
    cross_pool = cross_pool or {}

    # Gather candidate leaf values (tool_name, last_field, value)
    candidates: list[tuple[str, str, Any]] = []
    for entry in triple.tool_output:
        if not isinstance(entry, dict) or "name" not in entry:
            continue
        tool_name = entry["name"]
        results = entry.get("results", entry)
        for path, value in walk_leaves(results):
            if not path:
                continue
            field = path[-1]
            candidates.append((tool_name, field, value))

    rng.shuffle(candidates)

    for tool_name, field, value in candidates:
        # type-based, in priority of robustness
        if isinstance(value, bool):
            r = _try_bool_swap(value, answer, field)
            if r is not None:
                return r[0], [r[1]]
            continue
        if isinstance(value, int):
            r = _try_int_swap(value, answer, field, rng)
            if r is not None:
                return r[0], [r[1]]
        elif isinstance(value, float):
            r = _try_float_swap(value, answer, field, rng)
            if r is not None:
                return r[0], [r[1]]
        elif isinstance(value, str):
            if _is_url(value):
                r = _try_url_swap(value, answer, field, rng)
                if r is not None:
                    return r[0], [r[1]]
                continue
            # ISO date?
            if parse_iso(value) is not None:
                r = _try_date_swap(value, answer, field, rng)
                if r is not None:
                    return r[0], [r[1]]
                continue
            # categorical string in clean field — try in-sample pool, then cross-sample pool
            r = _try_pool_swap(value, answer, field, sample_pool.get((tool_name, field), set()),
                               rng, strategy="in_sample_pool")
            if r is not None:
                return r[0], [r[1]]
            r = _try_pool_swap(value, answer, field, cross_pool.get((tool_name, field), set()),
                               rng, strategy="cross_sample_pool")
            if r is not None:
                return r[0], [r[1]]

    return None

Run the rule-based pipeline. Each triple is tried against all candidate leaf values in a shuffled order; the first applicable strategy wins. Output: 751 / 1,034 = 72.6% coverage with zero broken spans.

In [ ]:
import random

rng = random.Random(42)
rule_records = []
strategy_counts = {}
for t in triples:
    result = inject(t, rng, cross_pool=cross_pool)
    if result is None:
        continue
    corrupted, spans = result
    for s in spans:
        strategy_counts[s.strategy] = strategy_counts.get(s.strategy, 0) + 1
    rule_records.append({
        "id": t.id,
        "query": t.user,
        "context": t.tool_output_raw,
        "output": corrupted,
        "original_output": t.assistant,
        "hallucination_labels": [
            {"start": s.start, "end": s.end, "text": s.text,
             "original_text": s.original_text, "field": s.field,
             "type": "Type1_Contradiction", "strategy": s.strategy}
            for s in spans
        ],
    })

print(f"Rule-based coverage: {len(rule_records)}/{len(triples)} = {100 * len(rule_records) / len(triples):.1f}%")
for k, v in sorted(strategy_counts.items(), key=lambda kv: -kv[1]):
    print(f"  {v:4d}  {k}")
write_jsonl(rule_records, DATA_DIR / "type1.jsonl")

## 4. Type 1: LLM fallback

For each triple not covered by the rule-based pipeline, one Qwen3-235B call is issued with a prompt that asks for a substring-level swap. The system prompt enforces word boundaries, case match, and avoids one specific grammar trap (`has been failed to X`). Each proposal is validated before application: `original_substring` must appear verbatim in the answer with intact word boundaries, `new_substring` must differ from the original.

In [ ]:
import requests

import json
import os
from dataclasses import dataclass
from typing import Any

import requests


OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
DEFAULT_MODEL = "qwen/qwen3-235b-a22b-2507"


SYSTEM_PROMPT = """You build datasets for testing hallucination detectors. Given a tool output (the ground truth) and an assistant answer based on it, you pick ONE specific fact in the answer that is grounded in the tool output, and propose a small substring-level edit that turns it into a contradiction (a hallucination).

Rules:
- Pick a single concrete factual span — a value, attribute, name, entity, label, status, count, etc.
- The "original_substring" MUST appear VERBATIM in the assistant answer.
- The "new_substring" must contradict what the tool output says about that fact.
- The "new_substring" must NOT itself appear in the tool output as a value.

GRAMMAR (very important):
- The substituted answer must read grammatically naturally. The edit must not leave dangling affixes or split a word.
- The "original_substring" must align with word boundaries — its first and last characters must be at word edges, not in the middle of a word. Example: if the answer says "successfully completed", do NOT pick "successful" (this would leave a stray "ly"); pick "successfully" itself, or pick "successfully completed".
- The "original_substring" must use the same CASE as in the answer (do not lowercase or capitalize).
- Match part of speech and grammatical role: noun phrase ↔ noun phrase, adjective ↔ adjective, verb form ↔ matching verb form.

CRITICAL grammar trap to avoid — passive voice + "failed to":
The phrase "has been / have been / was / were SUCCESSFULLY X-ed" cannot be edited into "has been failed to X" — that is ungrammatical. The auxiliary "has been" requires a past participle, not "failed to + verb".
Wrong: "has been successfully canceled" → "has been failed to cancel"
Right options:
  • "has been successfully canceled" → "has not been canceled"
  • "has been successfully canceled" → "could not be canceled"
  • "has been successfully canceled" → "was not canceled"
  • "successfully canceled" (as adverb+verb, without auxiliary) → "failed to cancel"
Same applies to "scheduled", "completed", "uploaded", "sent", "submitted", "simulated", etc. Always read the sentence aloud mentally and ensure the result is grammatical English.

- If you replace a value, keep surrounding syntax intact. The replacement should fit into the existing sentence structure.

OTHER CONSTRAINTS:
- Pick a fact that is GROUNDED in the tool output. Do not edit facts not supported by tool output.
- Avoid creating an obvious duplicate: if your replacement value is identical to a value appearing immediately adjacent in the answer (e.g. the previous row of a list already has it), pick a different replacement.

Return ONLY a JSON object on a single line, no markdown, no preamble."""


USER_PROMPT_TEMPLATE = """Tool output (JSON, ground truth):
{tool_output}

User query:
{user_query}

Assistant answer (clean):
{answer}

Return JSON: {{"original_substring": "...", "new_substring": "...", "reason": "..."}}"""


@dataclass
class LLMSwap:
    original_substring: str
    new_substring: str
    reason: str
    raw_response: str


class OpenRouterError(Exception):
    pass


def call_openrouter(messages: list[dict[str, str]], model: str = DEFAULT_MODEL,
                    api_key: str | None = None, temperature: float = 0.7,
                    timeout: float = 60.0) -> str:
    api_key = api_key or os.environ.get("OPENROUTER_API_KEY")
    if not api_key:
        raise OpenRouterError("OPENROUTER_API_KEY env var is not set")
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": model,
        "messages": messages,
        "temperature": temperature,
    }
    resp = requests.post(OPENROUTER_URL, headers=headers, json=payload, timeout=timeout)
    if resp.status_code != 200:
        raise OpenRouterError(f"HTTP {resp.status_code}: {resp.text[:500]}")
    data = resp.json()
    if "choices" not in data or not data["choices"]:
        raise OpenRouterError(f"No choices in response: {data}")
    return data["choices"][0]["message"]["content"]


def _extract_json(text: str) -> dict[str, Any]:
    """Pull the first JSON object out of a model response (in case of stray markdown fences etc.)."""
    text = text.strip()
    # Strip markdown fences if present
    if text.startswith("```"):
        text = text.split("```", 2)[1]
        if text.startswith("json"):
            text = text[4:]
        text = text.strip()
        if text.endswith("```"):
            text = text[: -3].strip()
    # Find first { … } object greedily
    start = text.find("{")
    end = text.rfind("}")
    if start < 0 or end <= start:
        raise ValueError(f"No JSON object found: {text[:200]!r}")
    return json.loads(text[start : end + 1])


def propose_swap(tool_output: str, user_query: str, answer: str,
                 model: str = DEFAULT_MODEL, api_key: str | None = None,
                 temperature: float = 0.7) -> LLMSwap:
    user_msg = USER_PROMPT_TEMPLATE.format(
        tool_output=tool_output,
        user_query=user_query,
        answer=answer,
    )
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_msg},
    ]
    raw = call_openrouter(messages, model=model, api_key=api_key, temperature=temperature)
    parsed = _extract_json(raw)
    return LLMSwap(
        original_substring=parsed.get("original_substring", ""),
        new_substring=parsed.get("new_substring", ""),
        reason=parsed.get("reason", ""),
        raw_response=raw,
    )

Run the LLM fallback with 10 parallel workers. Expected coverage: ~99% of the remaining triples, raising combined coverage to 99.8%.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

def _validate_swap(answer, orig, new):
    if not orig or not new or orig == new or orig not in answer:
        return False
    idx = answer.find(orig)
    left = answer[idx - 1] if idx > 0 else ""
    right = answer[idx + len(orig)] if idx + len(orig) < len(answer) else ""
    if left.isalpha() and orig[0].isalpha():
        return False
    if right.isalpha() and orig[-1].isalpha():
        return False
    return True

covered_ids = {r['id'] for r in rule_records}
uncovered = [t for t in triples if t.id not in covered_ids]
print(f'LLM fallback on {len(uncovered)} triples')

def _process(t):
    try:
        swap = propose_swap(tool_output=t.tool_output_raw, user_query=t.user, answer=t.assistant)
    except Exception:
        return None
    if not _validate_swap(t.assistant, swap.original_substring, swap.new_substring):
        return None
    idx = t.assistant.find(swap.original_substring)
    new_answer = t.assistant[:idx] + swap.new_substring + t.assistant[idx + len(swap.original_substring):]
    return {
        'id': t.id, 'query': t.user, 'context': t.tool_output_raw,
        'output': new_answer, 'original_output': t.assistant,
        'hallucination_labels': [{
            'start': idx, 'end': idx + len(swap.new_substring),
            'text': swap.new_substring, 'original_text': swap.original_substring,
            'field': '', 'type': 'Type1_Contradiction', 'strategy': 'llm',
            'llm_reason': swap.reason,
        }],
    }

llm_records = []
with ThreadPoolExecutor(max_workers=10) as pool:
    futures = {pool.submit(_process, t): t for t in uncovered}
    for fut in tqdm(as_completed(futures), total=len(futures), desc='Type1 LLM fallback'):
        r = fut.result()
        if r is not None:
            llm_records.append(r)

type1_full = rule_records + llm_records
print(f'Type 1 final coverage: {len(type1_full)}/{len(triples)} = {100*len(type1_full)/len(triples):.1f}%')
write_jsonl(type1_full, DATA_DIR / 'type1_full.jsonl')

## 5. Type 2: overgeneration injection

Each clean answer is rewritten by Qwen3-235B with one appended declarative sentence that introduces a fact, statistic or benchmark not present in the tool output. The model is instructed to make the fabrication topically natural and to avoid contradicting any specific value (which would belong to Type 1). The output is the appended sentence plus its start / end character offsets in the new answer.

In [ ]:
import json
from dataclasses import dataclass
from typing import Any



SYSTEM_PROMPT_T2 = """You build datasets for testing hallucination detectors. Specifically you create "overgeneration" hallucinations: an assistant adding extra factual claims, statistics, or background details to its answer that are NOT supported by the tool output (fabricated filler).

Given the tool output (ground truth), the user query, and the assistant's natural-language answer, produce ONE additional sentence to append at the end of the answer.

Rules for the appended sentence:
- It must be a DECLARATIVE STATEMENT — a fact, statistic, observation, recommendation, historical context, or trend claim.
- It must NOT be a question, offer, or "Would you like me to …" — those are a different hallucination type.
- The claim must be PLAUSIBLE-sounding and TOPICALLY natural: fit the domain, reference the discussed entities/values, sound like the kind of context a real assistant might add.
- The claim must NOT be supported by the tool output — invent new information (made-up numbers, attributed quotes, generic-sounding background, vague aggregate claims, etc.).
- Do NOT directly contradict a specific value in the tool output (that's a different hallucination type). Only add NEW information that the tool output is silent about.
- One single grammatical English sentence. 8-30 words. No markdown, no surrounding quotes.

Examples of good overgenerations:
- Tool says weather=sunny → "Such conditions typically persist for several days in this region during late spring."
- Tool returns top 5 artists → "Collectively, these artists have sold over 200 million records worldwide."
- Tool returns stock price → "Most analysts polled by Bloomberg expect the trend to continue through the next quarter."
- Tool returns valid VAT → "EU VAT databases are refreshed daily through official tax authority feeds."

Return ONLY a JSON object on one line:
{"sentence": "...", "claim": "<short label of the added claim>", "reason": "<one sentence: why this fact is not in the tool output>"}"""


USER_PROMPT_T2 = """Tool output (ground truth — the assistant CAN say these things):
{tool_output}

User query:
{user_query}

Assistant answer (current — no hallucinations yet):
{answer}

Now produce the JSON."""


@dataclass
class T2Proposal:
    sentence: str
    claim: str
    reason: str


def propose_type2(triple: Triple, api_key: str,
                  exclude_claims: list[str] | None = None,
                  model: str = "qwen/qwen3-235b-a22b-2507",
                  temperature: float = 0.7) -> T2Proposal | None:
    user_msg = USER_PROMPT_T2.format(
        tool_output=triple.tool_output_raw,
        user_query=triple.user,
        answer=triple.assistant,
    )
    if exclude_claims:
        exc = ", ".join(repr(c) for c in exclude_claims if c)
        user_msg += f"\n\nIMPORTANT: do NOT repeat or paraphrase any of these already-used claims: {exc}. Add a different kind of fabricated detail."
    try:
        raw = call_openrouter(
            [{"role": "system", "content": SYSTEM_PROMPT_T2},
             {"role": "user", "content": user_msg}],
            model=model, api_key=api_key, temperature=temperature,
        )
        parsed = _extract_json(raw)
    except (OpenRouterError, ValueError, json.JSONDecodeError):
        return None
    sentence = parsed.get("sentence", "").strip()
    claim = parsed.get("claim", "").strip()
    reason = parsed.get("reason", "").strip()
    if not sentence:
        return None
    return T2Proposal(sentence=sentence, claim=claim, reason=reason)


def make_record(triple: Triple, proposal: T2Proposal) -> dict[str, Any]:
    base = triple.assistant.rstrip()
    sep = " "
    new_answer = base + sep + proposal.sentence
    if len(triple.assistant) > len(base):
        new_answer = new_answer + triple.assistant[len(base):]
    start = len(base) + len(sep)
    end = start + len(proposal.sentence)
    return {
        "id": triple.id,
        "query": triple.user,
        "context": triple.tool_output_raw,
        "output": new_answer,
        "original_output": triple.assistant,
        "hallucination_labels": [{
            "start": start,
            "end": end,
            "text": proposal.sentence,
            "original_text": "",
            "field": proposal.claim,
            "type": "Type2_Overgeneration",
            "strategy": "llm",
            "llm_reason": proposal.reason,
        }],
    }

Train / val / test split is by source ID with 50 val + 150 test + remainder train. For train, three diverse variants per source (claim-exclusion to prevent repeats). For val / test, one variant per source. Wall-clock ~10 min with 15 workers.

In [ ]:
def split_triples(triples, n_val=50, n_test=150, seed=42):
    rng = random.Random(seed)
    ids = [t.id for t in triples]
    rng.shuffle(ids)
    test_ids = set(ids[:n_test])
    val_ids = set(ids[n_test:n_test + n_val])
    by_id = {t.id: t for t in triples}
    return ([by_id[i] for i in ids[n_test + n_val:]],
            [by_id[i] for i in val_ids],
            [by_id[i] for i in test_ids])

api_key = os.environ['OPENROUTER_API_KEY']
train_triples, val_triples, test_triples = split_triples(triples)
print(f'Split: train={len(train_triples)}, val={len(val_triples)}, test={len(test_triples)}')

def _process_t2(triple, target_n):
    records, used = [], []
    for vi in range(target_n):
        prop = propose_type2(triple, api_key=api_key, exclude_claims=used,
                              temperature=0.9 if vi > 0 else 0.7)
        if prop is None:
            continue
        rec = make_record(triple, prop)
        rec['id'] = f'{triple.id}_v{vi}'
        rec['source_id'] = triple.id
        records.append(rec)
        if prop.claim:
            used.append(prop.claim)
    return records

jobs = [(t, 'train', 3) for t in train_triples] + [(t, 'val', 1) for t in val_triples] + [(t, 'test', 1) for t in test_triples]
by_split = {'train': [], 'val': [], 'test': []}
with ThreadPoolExecutor(max_workers=15) as pool:
    futures = {pool.submit(_process_t2, t, n): (t, sp) for (t, sp, n) in jobs}
    for fut in tqdm(as_completed(futures), total=len(futures), desc='Type 2'):
        t, sp = futures[fut]
        for rec in fut.result():
            by_split[sp].append(rec)

for sp in ['train', 'val', 'test']:
    write_jsonl(by_split[sp], DATA_DIR / f'type2_{sp}.jsonl')
    print(f'  type2_{sp}: {len(by_split[sp])} samples')

## 6. Type 3: missing tool injection

The model receives the available-tools list and appends one sentence offering an action that requires a tool outside that list, anchored to entities in the dialogue. This is the only stage where the `tools_available` field matters at injection time; including it forces the LLM to produce subtle gaps (e.g. snowboard-availability tools without reservation → offer to reserve the gear).

In [ ]:
import json
from dataclasses import dataclass
from typing import Any



SYSTEM_PROMPT_T3 = """You build datasets for testing hallucination detectors. Specifically you create "missing tool" hallucinations: an assistant proposing a follow-up action that requires a capability NOT present in its available tool list.

Given the user query, the assistant's natural-language answer, and the list of tools the assistant had access to, produce ONE additional sentence to append at the end of the answer.

Rules for the appended sentence:
- It must SUGGEST a follow-up action (e.g. "Would you like me to …", "Shall I …", "I can also …", "Want me to …").
- The action must require a capability that is NOT covered by any of the available tools. Read tool names and descriptions to determine coverage.
- The action must be TOPICALLY natural for the conversation: fit the domain, reference the discussed entities/values, sound like a plausible next step a real assistant might offer.
- One single grammatical English sentence. 5-25 words. No markdown, no surrounding quotes.
- Do NOT propose a real-world action that the AVAILABLE tools clearly cover — that would not be a hallucination.

Return ONLY a JSON object on one line:
{"sentence": "...", "action": "<short label of the action>", "reason": "<one sentence: why the available tools don't cover this action>"}"""


USER_PROMPT_T3 = """Available tools (the assistant CAN do these):
{tools}

User query:
{user_query}

Assistant answer:
{answer}

Now produce the JSON."""


@dataclass
class T3Proposal:
    sentence: str
    action: str
    reason: str


def _format_tools(tools: list[dict[str, Any]]) -> str:
    if not tools:
        return "(no tools)"
    lines = []
    for t in tools:
        if not isinstance(t, dict):
            continue
        name = t.get("name", "?")
        desc = t.get("description", "")
        lines.append(f"- {name}: {desc[:200]}")
    return "\n".join(lines)


def propose_type3(triple: Triple, api_key: str,
                  exclude_actions: list[str] | None = None,
                  model: str = "qwen/qwen3-235b-a22b-2507",
                  temperature: float = 0.7) -> T3Proposal | None:
    user_msg = USER_PROMPT_T3.format(
        tools=_format_tools(triple.tools_available),
        user_query=triple.user,
        answer=triple.assistant,
    )
    if exclude_actions:
        exc = ", ".join(repr(a) for a in exclude_actions if a)
        user_msg += f"\n\nIMPORTANT: do NOT propose any of these already-used actions: {exc}. Pick a different capability."
    try:
        raw = call_openrouter(
            [{"role": "system", "content": SYSTEM_PROMPT_T3},
             {"role": "user", "content": user_msg}],
            model=model, api_key=api_key, temperature=temperature,
        )
        parsed = _extract_json(raw)
    except (OpenRouterError, ValueError, json.JSONDecodeError):
        return None
    sentence = parsed.get("sentence", "").strip()
    action = parsed.get("action", "").strip()
    reason = parsed.get("reason", "").strip()
    if not sentence:
        return None
    return T3Proposal(sentence=sentence, action=action, reason=reason)


def make_record(triple: Triple, proposal: T3Proposal) -> dict[str, Any]:
    base = triple.assistant.rstrip()
    sep = " "
    new_answer = base + sep + proposal.sentence
    if len(triple.assistant) > len(base):
        new_answer = new_answer + triple.assistant[len(base):]
    start = len(base) + len(sep)
    end = start + len(proposal.sentence)
    return {
        "id": triple.id,
        "query": triple.user,
        "context": triple.tool_output_raw,
        "tools_available": triple.tools_available,
        "output": new_answer,
        "original_output": triple.assistant,
        "hallucination_labels": [{
            "start": start,
            "end": end,
            "text": proposal.sentence,
            "original_text": "",
            "field": proposal.action,
            "type": "Type3_MissingTool",
            "strategy": "llm",
            "llm_reason": proposal.reason,
        }],
    }

In [ ]:
def _process_t3(triple, target_n):
    records, used = [], []
    for vi in range(target_n):
        prop = propose_type3(triple, api_key=api_key, exclude_actions=used,
                              temperature=0.9 if vi > 0 else 0.7)
        if prop is None:
            continue
        rec = make_record(triple, prop)
        rec['id'] = f'{triple.id}_v{vi}'
        rec['source_id'] = triple.id
        records.append(rec)
        if prop.action:
            used.append(prop.action)
    return records

jobs = [(t, 'train', 3) for t in train_triples] + [(t, 'val', 1) for t in val_triples] + [(t, 'test', 1) for t in test_triples]
by_split = {'train': [], 'val': [], 'test': []}
with ThreadPoolExecutor(max_workers=15) as pool:
    futures = {pool.submit(_process_t3, t, n): (t, sp) for (t, sp, n) in jobs}
    for fut in tqdm(as_completed(futures), total=len(futures), desc='Type 3'):
        t, sp = futures[fut]
        for rec in fut.result():
            by_split[sp].append(rec)

for sp in ['train', 'val', 'test']:
    write_jsonl(by_split[sp], DATA_DIR / f'type3_{sp}.jsonl')
    print(f'  type3_{sp}: {len(by_split[sp])} samples')

## 7. Type 1 augmentation

For each train source triple, generate up to five distinct rule-based candidates (round-robin across strategies for diversity, prefer unseen fields). If fewer than five rule-based candidates are available, fill with LLM calls at higher temperature, excluding already-used `original_substring`s. Final train: 4,057 samples (target 5 per source, actual average 4.86).

In [ ]:
import json
import random
import re
from collections import defaultdict
from dataclasses import dataclass
from typing import Any



@dataclass
class Variant:
    output: str
    span: Span


def select_diverse(candidates: list[tuple[str, Span]], n: int,
                   rng: random.Random) -> list[Variant]:
    """Round-robin pick up to n candidates, preferring strategy diversity, then field diversity."""
    if not candidates:
        return []
    by_strategy: dict[str, list[tuple[str, Span]]] = defaultdict(list)
    for c in candidates:
        by_strategy[c[1].strategy].append(c)
    for items in by_strategy.values():
        rng.shuffle(items)
    strategies = list(by_strategy.keys())
    rng.shuffle(strategies)
    chosen: list[Variant] = []
    used_fields: set[str] = set()
    # First pass: one per strategy, preferring unseen fields
    while len(chosen) < n and any(by_strategy[s] for s in strategies):
        progressed = False
        for s in strategies:
            if not by_strategy[s] or len(chosen) >= n:
                continue
            # prefer a candidate with an unused field
            chosen_idx = None
            for i, c in enumerate(by_strategy[s]):
                if c[1].field not in used_fields:
                    chosen_idx = i
                    break
            if chosen_idx is None:
                chosen_idx = 0
            text, span = by_strategy[s].pop(chosen_idx)
            chosen.append(Variant(output=text, span=span))
            used_fields.add(span.field)
            progressed = True
        if not progressed:
            break
    return chosen[:n]


def _validate_swap(answer: str, orig: str, new: str) -> bool:
    if not orig or not new or orig == new:
        return False
    if orig not in answer:
        return False
    idx = answer.find(orig)
    left = answer[idx - 1] if idx > 0 else ""
    right = answer[idx + len(orig)] if idx + len(orig) < len(answer) else ""
    if left.isalpha() and orig[0].isalpha():
        return False
    if right.isalpha() and orig[-1].isalpha():
        return False
    return True


def llm_propose_excluding(triple: Triple, exclude_originals: list[str],
                          api_key: str, temperature: float = 0.9,
                          model: str = "qwen/qwen3-235b-a22b-2507") -> Variant | None:
    extra = ""
    if exclude_originals:
        exc = ", ".join(repr(e) for e in exclude_originals)
        extra = f"\n\nIMPORTANT: choose a DIFFERENT fact than these already-used spans: {exc}"
    user_msg = USER_PROMPT_TEMPLATE.format(
        tool_output=triple.tool_output_raw,
        user_query=triple.user,
        answer=triple.assistant,
    ) + extra
    try:
        raw = call_openrouter(
            [{"role": "system", "content": SYSTEM_PROMPT},
             {"role": "user", "content": user_msg}],
            model=model, api_key=api_key, temperature=temperature,
        )
        parsed = _extract_json(raw)
    except (OpenRouterError, ValueError, json.JSONDecodeError):
        return None

    orig = parsed.get("original_substring", "")
    new = parsed.get("new_substring", "")
    reason = parsed.get("reason", "")
    if not _validate_swap(triple.assistant, orig, new):
        return None
    idx = triple.assistant.find(orig)
    new_answer = triple.assistant[:idx] + new + triple.assistant[idx + len(orig):]
    span = Span(start=idx, end=idx + len(new), text=new,
                original_text=orig, field="", strategy="llm")
    return Variant(output=new_answer, span=span)


def variant_to_record(triple: Triple, variant: Variant, variant_idx: int) -> dict[str, Any]:
    return {
        "id": f"{triple.id}_v{variant_idx}",
        "source_id": triple.id,
        "query": triple.user,
        "context": triple.tool_output_raw,
        "output": variant.output,
        "original_output": triple.assistant,
        "hallucination_labels": [
            {
                "start": variant.span.start,
                "end": variant.span.end,
                "text": variant.span.text,
                "original_text": variant.span.original_text,
                "field": variant.span.field,
                "type": "Type1_Contradiction",
                "strategy": variant.span.strategy,
            }
        ],
    }

In [ ]:
TARGET_VARIANTS = 5
augmented_train = []
for source in tqdm(train_triples, desc='Type 1 augment'):
    candidates = collect_all_swaps(source, rng, cross_pool=cross_pool)
    variants = select_diverse(candidates, TARGET_VARIANTS, rng)
    used = [v.span.original_text for v in variants]
    attempts_left = max(0, TARGET_VARIANTS - len(variants))
    while attempts_left > 0:
        v = llm_propose_excluding(source, used, api_key=api_key, temperature=0.9)
        if v is None:
            attempts_left -= 1
            continue
        variants.append(v)
        used.append(v.span.original_text)
        attempts_left -= 1
        if len(variants) >= TARGET_VARIANTS:
            break
    for i, v in enumerate(variants):
        augmented_train.append(variant_to_record(source, v, i))

augmented_val = [r for r in rule_records + llm_records if r['id'] in {t.id for t in val_triples}]
for r in augmented_val:
    r['source_id'] = r['id']
    r['id'] = f"{r['id']}_v0"
augmented_test = [r for r in rule_records + llm_records if r['id'] in {t.id for t in test_triples}]
for r in augmented_test:
    r['source_id'] = r['id']
    r['id'] = f"{r['id']}_v0"

write_jsonl(augmented_train, DATA_DIR / 'type1_train_balanced.jsonl')
write_jsonl(augmented_val, DATA_DIR / 'type1_val_balanced.jsonl')
write_jsonl(augmented_test, DATA_DIR / 'type1_test_balanced.jsonl')
print(f'type1_train_balanced: {len(augmented_train)}')
print(f'type1_val_balanced: {len(augmented_val)}')
print(f'type1_test_balanced: {len(augmented_test)}')

## 8. Combine splits with clean samples

Per-type splits are concatenated with type-prefixed IDs (`t1_…`, `t2_…`, `t3_…`) to prevent cross-type ID collisions when a single source triple was used in all three types. Each split is balanced by adding clean samples (unmodified ToolACE responses with empty `hallucination_labels`); clean IDs receive a `_clean` suffix. The combined test set has 25% clean samples — without them, example precision would be trivially 1.0.

In [ ]:
TRIPLES_BY_ID = {t.id: t for t in triples}

def make_clean(source_id, split_id):
    t = TRIPLES_BY_ID[source_id]
    return {
        'id': split_id, 'source_id': source_id,
        'query': t.user, 'context': t.tool_output_raw,
        'tools_available': t.tools_available,
        'output': t.assistant, 'hallucination_labels': [],
    }

def with_tools(rec):
    if 'tools_available' not in rec:
        sid = rec.get('source_id') or rec['id'].rsplit('_v', 1)[0]
        if sid in TRIPLES_BY_ID:
            rec = dict(rec)
            rec['tools_available'] = TRIPLES_BY_ID[sid].tools_available
    return rec

def combine(per_type_records, prefix, split_name, add_clean_for):
    out = []
    for r in per_type_records:
        r = with_tools(r)
        r = dict(r)
        r['id'] = f'{prefix}_{r["id"]}'
        out.append(r)
    for source_id in add_clean_for:
        out.append(make_clean(source_id, f'{source_id}_clean'))
    return out

type2_train = read_jsonl(DATA_DIR / 'type2_train.jsonl')
type2_val   = read_jsonl(DATA_DIR / 'type2_val.jsonl')
type2_test  = read_jsonl(DATA_DIR / 'type2_test.jsonl')
type3_train = read_jsonl(DATA_DIR / 'type3_train.jsonl')
type3_val   = read_jsonl(DATA_DIR / 'type3_val.jsonl')
type3_test  = read_jsonl(DATA_DIR / 'type3_test.jsonl')

train_clean_ids = [t.id for t in train_triples]
val_clean_ids = [t.id for t in val_triples]
test_clean_ids = [t.id for t in test_triples]

combined_train = (
    combine(augmented_train, 't1', 'train', set())
    + combine(type2_train, 't2', 'train', set())
    + combine(type3_train, 't3', 'train', set())
    + [make_clean(s, f'{s}_clean') for s in train_clean_ids]
)
combined_val = (
    combine(augmented_val, 't1', 'val', set())
    + combine(type2_val, 't2', 'val', set())
    + combine(type3_val, 't3', 'val', set())
    + [make_clean(s, f'{s}_clean') for s in val_clean_ids]
)
combined_test = (
    combine(augmented_test, 't1', 'test', set())
    + combine(type2_test, 't2', 'test', set())
    + combine(type3_test, 't3', 'test', set())
    + [make_clean(s, f'{s}_clean') for s in test_clean_ids]
)

write_jsonl(combined_train, DATA_DIR / 'combined_train.jsonl')
write_jsonl(combined_val, DATA_DIR / 'combined_val.jsonl')
write_jsonl(combined_test, DATA_DIR / 'combined_test.jsonl')
print(f'combined_train: {len(combined_train)}')
print(f'combined_val:   {len(combined_val)}')
print(f'combined_test:  {len(combined_test)}')

## 9. (Optional) Upload to Hugging Face

Push the combined splits to a HF dataset repository. Requires `HF_TOKEN` or `huggingface-cli login`.

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
HF_REPO = "dakoblov/Hallucinations"
for name in ["combined_train.jsonl", "combined_val.jsonl", "combined_test.jsonl"]:
    api.upload_file(
        path_or_fileobj=str(DATA_DIR / name),
        path_in_repo=name,
        repo_id=HF_REPO,
        repo_type="dataset",
    )
print(f"Uploaded to https://huggingface.co/datasets/{HF_REPO}")